##Project Name: Customer Satisfaction (CSAT) scores using Deep Learning Artificial Neural Networks (ANN)

##Overview:
This project focuses on predicting Customer Satisfaction (CSAT) scores using Deep Learning Artificial Neural Networks (ANN). In the context of e-commerce, understanding customer satisfaction through their interactions and feedback is crucial for enhancing service quality, customer retention, and overall business growth. By leveraging advanced neural network models, we aim to accurately forecast CSAT scores based on a myriad of interaction-related features, providing actionable insights for service improvement.

##Project Goal:  
The primary goal of this project is to develop a deep learning model that can accurately predict the CSAT scores based on customer interactions and feedback. By doing so, we aim to provide e-commerce businesses with a powerful tool to monitor and enhance customer satisfaction in real-time, thereby improving service quality and fostering customer loyalty.

##Project Summary:
This project focuses on predicting Customer Satisfaction (CSAT) in an e-commerce platform using Artificial Neural Networks (ANN). CSAT is an important metric that helps businesses understand customer experience and improve their services. The dataset includes features such as delivery time, product details, pricing, and return status. Data preprocessing was performed to handle missing values and convert categorical data into numerical form. Feature engineering was applied to create meaningful variables like delivery delay and return indicators. An ANN model was developed to learn patterns between input features and CSAT scores and was trained using historical data. The model’s performance was evaluated using metrics such as accuracy, precision, and recall. Based on the results, key insights were generated to identify factors affecting customer satisfaction. Finally, the trained model was deployed locally to predict CSAT for new data and support better business decision-making.

#Git-Hub Link:

#Let's Begin:

#### First install necessery libraries:

In [1]:
!pip install tensorflow

###1) Import Libraries:

####I imported libraries for data processing, visualization, and building the ANN model.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

###2) Load Dataset:

####Loaded dataset and viewed initial records to understand structure.

In [3]:
df = pd.read_csv('/content/eCommerce_Customer_support_data.csv')


In [4]:
print(df.head())

                              Unique id channel_name         category  \
0  7e9ae164-6a8b-4521-a2d4-58f7c9fff13f      Outcall  Product Queries   
1  b07ec1b0-f376-43b6-86df-ec03da3b2e16      Outcall  Product Queries   
2  200814dd-27c7-4149-ba2b-bd3af3092880      Inbound    Order Related   
3  eb0d3e53-c1ca-42d3-8486-e42c8d622135      Inbound          Returns   
4  ba903143-1e54-406c-b969-46c52f92e5df      Inbound     Cancellation   

                   Sub-category Customer Remarks  \
0                Life Insurance              NaN   
1  Product Specific Information              NaN   
2             Installation/demo              NaN   
3        Reverse Pickup Enquiry              NaN   
4                    Not Needed              NaN   

                               Order_id order_date_time Issue_reported at  \
0  c27c9bb4-fa36-4140-9f1f-21009254ffdb             NaN  01/08/2023 11:13   
1  d406b0c7-ce17-4654-b9de-f08d421254bd             NaN  01/08/2023 12:52   
2  c273368d-b961-

###3) Basic Exploration:

####Checked data types, missing values, and statistical summary, and shape of the dataset.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85907 entries, 0 to 85906
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Unique id                85907 non-null  object 
 1   channel_name             85907 non-null  object 
 2   category                 85907 non-null  object 
 3   Sub-category             85907 non-null  object 
 4   Customer Remarks         28742 non-null  object 
 5   Order_id                 67675 non-null  object 
 6   order_date_time          17214 non-null  object 
 7   Issue_reported at        85907 non-null  object 
 8   issue_responded          85907 non-null  object 
 9   Survey_response_Date     85907 non-null  object 
 10  Customer_City            17079 non-null  object 
 11  Product_category         17196 non-null  object 
 12  Item_price               17206 non-null  float64
 13  connected_handling_time  242 non-null    float64
 14  Agent_name            

In [6]:
df.isnull().sum()

,0
Unique id,0
channel_name,0
category,0
Sub-category,0
Customer Remarks,57165
Order_id,18232
order_date_time,68693
Issue_reported at,0
issue_responded,0
Survey_response_Date,0


In [7]:
df.describe()

,Item_price,connected_handling_time,CSAT Score
count,17206.000000,242.000000,85907.000000
mean,5660.774846,462.400826,4.242157
std,12825.728411,246.295037,1.378903
min,0.000000,0.000000,1.000000
25%,392.000000,293.000000,4.000000
50%,979.000000,427.000000,5.000000
75%,2699.750000,592.250000,5.000000
max,164999.000000,1986.000000,5.000000


In [8]:
df.shape

(85907, 20)

####The dataset contains 85,907 observations and 20 features, providing sufficient data for training a deep learning model.

####I explored the dataset using functions like df.info() and df.isnull().sum(). The dataset has around 85,000 records with different types of features. I found missing values in several columns, so instead of removing data, I handled them using imputation techniques. This helped in retaining more data and improving model performance.

###4) Handle Missing Values:

####I handled missing values by removing highly sparse columns, imputing categorical features with placeholders, and using median imputation for numerical values.

In [9]:
# Drop unnecessary columns
df.drop(['Unique id', 'Agent_name' , 'Supervisor' , 'Manager'], axis=1, inplace=True)

# Drop columns with too many missing values
df.drop(['connected_handling_time', 'order_date_time'], axis=1, inplace=True)

# Drop rows where target is missing
df = df.dropna(subset=['CSAT Score'])

In [10]:

# Fill categorical columns
df['Customer Remarks'] = df['Customer Remarks'].fillna("No Remark")
df['Customer_City'] = df['Customer_City'].fillna("Unknown")
df['Product_category'] = df['Product_category'].fillna("Unknown")

In [11]:
# Fill numerical columns
df['Item_price'] = df['Item_price'].fillna(df['Item_price'].median())

# Drop remaining small missing values
df.dropna(inplace=True)


In [12]:
print(df.head())

  channel_name         category                  Sub-category  \
0      Outcall  Product Queries                Life Insurance   
1      Outcall  Product Queries  Product Specific Information   
2      Inbound    Order Related             Installation/demo   
3      Inbound          Returns        Reverse Pickup Enquiry   
4      Inbound     Cancellation                    Not Needed   

  Customer Remarks                              Order_id Issue_reported at  \
0        No Remark  c27c9bb4-fa36-4140-9f1f-21009254ffdb  01/08/2023 11:13   
1        No Remark  d406b0c7-ce17-4654-b9de-f08d421254bd  01/08/2023 12:52   
2        No Remark  c273368d-b961-44cb-beaf-62d6fd6c00d5  01/08/2023 20:16   
3        No Remark  5aed0059-55a4-4ec6-bb54-97942092020a  01/08/2023 20:56   
4        No Remark  e8bed5a9-6933-4aff-9dc6-ccefd7dcde59  01/08/2023 10:30   

    issue_responded Survey_response_Date Customer_City Product_category  \
0  01/08/2023 11:47            01-Aug-23       Unknown          U

###Convert Date Columns:

####I converted multiple columns into datetime format using a loop. Since the dataset had dates in DD/MM/YYYY format, I used dayfirst=True to ensure correct parsing. I also used errors='coerce' to handle invalid or inconsistent values by converting them into missing values.

In [13]:
date_cols = ['Issue_reported at', 'issue_responded', 'Survey_response_Date']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

print(df.head())


  channel_name         category                  Sub-category  \
0      Outcall  Product Queries                Life Insurance   
1      Outcall  Product Queries  Product Specific Information   
2      Inbound    Order Related             Installation/demo   
3      Inbound          Returns        Reverse Pickup Enquiry   
4      Inbound     Cancellation                    Not Needed   

  Customer Remarks                              Order_id   Issue_reported at  \
0        No Remark  c27c9bb4-fa36-4140-9f1f-21009254ffdb 2023-08-01 11:13:00   
1        No Remark  d406b0c7-ce17-4654-b9de-f08d421254bd 2023-08-01 12:52:00   
2        No Remark  c273368d-b961-44cb-beaf-62d6fd6c00d5 2023-08-01 20:16:00   
3        No Remark  5aed0059-55a4-4ec6-bb54-97942092020a 2023-08-01 20:56:00   
4        No Remark  e8bed5a9-6933-4aff-9dc6-ccefd7dcde59 2023-08-01 10:30:00   

      issue_responded Survey_response_Date Customer_City Product_category  \
0 2023-08-01 11:47:00           2023-08-01       Un

/tmp/ipykernel_8846/4238291304.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')


In [14]:
#Create Response Time Feature
df['response_time'] = (df['issue_responded'] - df['Issue_reported at']).dt.total_seconds()
df.head()

,channel_name,category,Sub-category,Customer Remarks,Order_id,Issue_reported at,issue_responded,Survey_response_Date,Customer_City,Product_category,Item_price,Tenure Bucket,Agent Shift,CSAT Score,response_time
0,Outcall,Product Queries,Life Insurance,No Remark,c27c9bb4-fa36-4140-9f1f-21009254ffdb,2023-08-01 11:13:00,2023-08-01 11:47:00,2023-08-01,Unknown,Unknown,979.0,On Job Training,Morning,5,2040.0
1,Outcall,Product Queries,Product Specific Information,No Remark,d406b0c7-ce17-4654-b9de-f08d421254bd,2023-08-01 12:52:00,2023-08-01 12:54:00,2023-08-01,Unknown,Unknown,979.0,>90,Morning,5,120.0
2,Inbound,Order Related,Installation/demo,No Remark,c273368d-b961-44cb-beaf-62d6fd6c00d5,2023-08-01 20:16:00,2023-08-01 20:38:00,2023-08-01,Unknown,Unknown,979.0,On Job Training,Evening,5,1320.0
3,Inbound,Returns,Reverse Pickup Enquiry,No Remark,5aed0059-55a4-4ec6-bb54-97942092020a,2023-08-01 20:56:00,2023-08-01 21:16:00,2023-08-01,Unknown,Unknown,979.0,>90,Evening,5,1200.0
4,Inbound,Cancellation,Not Needed,No Remark,e8bed5a9-6933-4aff-9dc6-ccefd7dcde59,2023-08-01 10:30:00,2023-08-01 10:32:00,2023-08-01,Unknown,Unknown,979.0,0-30,Morning,5,120.0


####Created response time feature, which is a key factor affecting customer satisfaction.

####DROP datetime columns:

In [15]:
df.drop(['Issue_reported at', 'issue_responded', 'Survey_response_Date'], axis=1, inplace=True)

In [16]:
#Text Length (from Customer Remarks)
df['remark_length'] = df['Customer Remarks'].apply(len)
df.head()

,channel_name,category,Sub-category,Customer Remarks,Order_id,Customer_City,Product_category,Item_price,Tenure Bucket,Agent Shift,CSAT Score,response_time,remark_length
0,Outcall,Product Queries,Life Insurance,No Remark,c27c9bb4-fa36-4140-9f1f-21009254ffdb,Unknown,Unknown,979.0,On Job Training,Morning,5,2040.0,9
1,Outcall,Product Queries,Product Specific Information,No Remark,d406b0c7-ce17-4654-b9de-f08d421254bd,Unknown,Unknown,979.0,>90,Morning,5,120.0,9
2,Inbound,Order Related,Installation/demo,No Remark,c273368d-b961-44cb-beaf-62d6fd6c00d5,Unknown,Unknown,979.0,On Job Training,Evening,5,1320.0,9
3,Inbound,Returns,Reverse Pickup Enquiry,No Remark,5aed0059-55a4-4ec6-bb54-97942092020a,Unknown,Unknown,979.0,>90,Evening,5,1200.0,9
4,Inbound,Cancellation,Not Needed,No Remark,e8bed5a9-6933-4aff-9dc6-ccefd7dcde59,Unknown,Unknown,979.0,0-30,Morning,5,120.0,9


###Encode Categorical Features:

In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])
df.head()


,channel_name,category,Sub-category,Customer Remarks,Order_id,Customer_City,Product_category,Item_price,Tenure Bucket,Agent Shift,CSAT Score,response_time,remark_length
0,2,8,19,7276,51500,1722,9,979.0,4,2,5,2040.0,9
1,2,8,34,7276,56068,1722,9,979.0,3,2,5,120.0,9
2,1,5,15,7276,51493,1722,9,979.0,4,1,5,1320.0,9
3,1,10,40,7276,24227,1722,9,979.0,3,1,5,1200.0,9
4,1,1,22,7276,61514,1722,9,979.0,0,2,5,120.0,9


####I encode the specific categorical column to numerical.

###Define Features & Target:

#####I separated the dataset into input features and target variable. The independent variables (X) contain all the features used for prediction, while the dependent variable (y) is the CSAT score, which the model aims to predict.

In [18]:
X = df.drop('CSAT Score', axis=1)
y = df['CSAT Score']

###Train-Test Split:

####I split the dataset into training and testing sets using an 80-20 ratio. The training set is used to train the model, while the testing set is used to evaluate its performance on unseen data.

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

###Feature Scaling:

#####I applied feature scaling using StandardScaler to normalize the input features so that all variables have a similar scale. This helps improve the performance and convergence of the ANN model.

In [20]:
type(X_train)

pandas.core.frame.DataFrame

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)




####Normalized features to improve ANN performance.

###Build ANN Model:

1) Initialize Model: Create an instance of a Sequential model. This allows you to build the ANN as a linear stack of layers.



In [22]:
model = Sequential()

2.Add Layers: Add an input layer, one or more hidden layers, and an output layer.

In [23]:

model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='linear'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Compile the Model


####I compiled the ANN model using the Adam optimizer for efficient learning, mean squared error as the loss function since it is a regression problem, and mean absolute error as an evaluation metric to measure prediction accuracy.

In [24]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

###Train Model:

Fit Model to Data: Use the fit() method to train the model on your training data. Set epochs and batch size.

Trained ANN model using training data to learn patterns.”

In [25]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
1354/1354 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 2.4125 - mae: 1.1610 - val_loss: 1.8580 - val_mae: 1.0035
Epoch 2/10
1354/1354 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.8362 - mae: 1.0176 - val_loss: 1.8533 - val_mae: 1.0764
Epoch 3/10
1354/1354 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.8041 - mae: 1.0086 - val_loss: 1.8180 - val_mae: 1.0493
Epoch 4/10
1354/1354 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.7958 - mae: 1.0063 - val_loss: 1.8269 - val_mae: 0.9763
Epoch 5/10
1354/1354 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.7802 - mae: 1.0022 - val_loss: 1.7934 - val_mae: 1.0213
Epoch 6/10
1354/1354 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.7777 - mae: 1.0020 - val_loss: 1.7941 - val_mae: 0.9860
Epoch 7/10
1354/1354 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.7707 - mae: 0.9990 - val_loss: 1.8036 - val_mae: 1.0566
Epoch 8/10
1354/1354 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.7676 - mae: 0.9975 - val_loss: 1.7900 - val_mae: 1.0309
Epoch 9/10
1354/1354 ━━━━━━━━━━━━━━━━━━━

####I trained the ANN model using the training dataset for 10 epochs with a batch size of 32. I also used 20% of the training data as validation data to monitor the model’s performance during training and avoid overfitting.

###Evaluate the Model:

Test Data Evaluation: Assess the model's performance on the test dataset to see how well it generalizes.



In [26]:
loss, mae = model.evaluate(X_test, y_test)
print("MAE:", mae)

423/423 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.7996 - mae: 1.0967
MAE: 1.0967140197753906


####I evaluated the trained ANN model on the test dataset using Mean Squared Error as the loss function and Mean Absolute Error as the performance metric. The model achieved an MAE of approximately 1.09, which indicates that the predicted CSAT scores are on average less than 1 unit away from the actual values.  
The model achieved an MAE of 1.09 , indicating good prediction accuracy with less than one-point deviation.

###Predictions:

I trained model to predict CSAT scores for the test data.

In [27]:
y_pred = model.predict(X_test)

423/423 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [28]:
print(y_pred[:5])

[[4.2261553]
 [2.6997855]
 [4.330544 ]
 [3.800642 ]
 [3.3626695]]


##### My model is predicting values like:

Around 4 → satisfied customers  

Around 2–3 → unhappy customers

###Insight Generation:

In [30]:
#Compare with actual values
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

print(results.head())

       Actual  Predicted
51660       5   4.226155
1945        1   2.699785
34369       2   4.330544
48185       5   3.800642
3559        5   3.362669


#####The predicted CSAT scores were compared with actual values using a DataFrame. It was observed that the model is able to capture general trends in customer satisfaction, but there are some deviations in predictions, especially for extreme values such as very low or very high CSAT scores. This indicates that while the model performs reasonably well overall, there is scope for improvement in accurately predicting edge cases.

In [31]:
#Error Analysis
results['Error'] = abs(results['Actual'] - results['Predicted'])
print(results.head())

       Actual  Predicted     Error
51660       5   4.226155  0.773845
1945        1   2.699785  1.699785
34369       2   4.330544  2.330544
48185       5   3.800642  1.199358
3559        5   3.362669  1.637331


##Local Deployment:

####This stores your trained ANN.

In [34]:
#Save your model
model.save("csat_model.keras")

In [35]:
#Load model later
from tensorflow.keras.models import load_model
loaded_model = load_model("csat_model.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [36]:
#Take one sample input
new_data = X_test[0].reshape(1, -1)

In [38]:

import pandas as pd

new_data = pd.DataFrame([X_test[0]], columns=X.columns)

new_data = scaler.transform(new_data)

In [39]:
#Predict
prediction = loaded_model.predict(new_data)

print("Predicted CSAT:", prediction[0][0])
print("Actual CSAT:", y_test.iloc[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
Predicted CSAT: 4.088925
Actual CSAT: 5


####The model predicted a CSAT score close to the actual value, showing that it can generalize well to new data, although slight deviations are expected.

####The trained model was deployed locally by saving and loading it for future use. A sample input was provided to the model, and it successfully predicted the CSAT score. This demonstrates how the model can be used in real-world scenarios to estimate customer satisfaction.

####Deployment = Save model + Load model + Predict on new data

#Conclusion :  
n this project, an end-to-end machine learning pipeline was developed to predict Customer Satisfaction (CSAT) scores using an Artificial Neural Network (ANN). The dataset was first cleaned and preprocessed by handling missing values, converting date features, and encoding categorical variables. Relevant features were engineered to improve the model’s predictive capability.

The ANN model was trained on the processed data and evaluated using Mean Absolute Error (MAE), where it achieved reasonable performance, indicating that the model can effectively capture general patterns in customer satisfaction. Although the model performs well for average cases, it shows some limitations in predicting extreme values, particularly very low CSAT scores.

Further analysis of predictions revealed meaningful insights, such as the model’s tendency to slightly overestimate dissatisfaction and underestimate high satisfaction in certain cases. This highlights opportunities for improvement through better feature engineering and model tuning.

Finally, the model was successfully deployed in a local environment by saving and loading it for real-time predictions. This demonstrates its practical usability in real-world customer support systems, where it can help organizations proactively identify customer satisfaction levels and improve service quality.

